# Mission 3 P0 baseline local re-run (TF-IDF + OvR Logistic Regression)

Local (Windows) adaptation of the Colab draft. Executed with `jupyter nbconvert --execute`; cell outputs are the submission log.
Reference: Macro F1 0.5676 (tol ±0.003), N=29,200, 0 load failures.

In [ ]:
# Cell 0: config + provenance
import os, sys, platform, hashlib, subprocess, json, time
from pathlib import Path

os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUNBUFFERED"] = "1"

REPO = Path(r"C:\Users\<user>\Desktop\아주대\dcc\01_prelim\dcc2026-prelim")
LABEL_DIR = REPO / "data" / "raw_merged" / "Training" / "2.라벨링데이터" / "TL_서울_구급"
SCRIPT = REPO / "scripts" / "mission3_baseline 최종 수정본.py"
OUT = REPO / "outputs" / "m3_p0_rerun"
OUT.mkdir(parents=True, exist_ok=True)
EXPECTED_N = 29200
REF_MACRO_F1 = 0.5676
TOL = 0.003

def git(*args):
    return subprocess.run(["git", "-C", str(REPO), *args], capture_output=True,
                          text=True, encoding="utf-8", errors="replace").stdout.strip()

def sha256(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

HEAD = git("rev-parse", "HEAD")
print("git HEAD          :", HEAD)
print("git log -1        :", git("log", "-1", "--oneline"))
st = git("status", "--porcelain", "--", "scripts/")
print("git status scripts:", st if st else "(clean)")
SCRIPT_SHA = sha256(SCRIPT)
print("script            :", SCRIPT)
print("script sha256     :", SCRIPT_SHA)
print("python            :", sys.version.replace("\n", " "))
print("executable        :", sys.executable)
print("platform          :", platform.platform(), "|", platform.processor())
print("LABEL_DIR         :", LABEL_DIR)
n_json = sum(1 for _ in LABEL_DIR.rglob("*.json"))
print("JSON count        :", n_json)
assert n_json == EXPECTED_N, f"expected {EXPECTED_N} JSON, got {n_json}"
print("OUT               :", OUT)


In [ ]:
# Cell 1: dependency check (local: install iterative-stratification only if missing)
try:
    import iterstrat
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "iterative-stratification"], check=True)
    import iterstrat
import sklearn, numpy, pandas, scipy, joblib
print("iterstrat", getattr(iterstrat, "__version__", "(no __version__)"), "|", iterstrat.__file__)
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| scipy", scipy.__version__, "| joblib", joblib.__version__)


In [ ]:
# Cell 2: streaming run() helper
LOG = OUT / "run_log.txt"

def run(args):
    cmd = [sys.executable, "-u", *map(str, args)]
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    print("$", " ".join(f'"{c}"' if " " in c else c for c in cmd), flush=True)
    t0 = time.time()
    with open(LOG, "a", encoding="utf-8") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                env=env, text=True, encoding="utf-8", errors="replace",
                                bufsize=1, cwd=str(REPO))
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        rc = proc.wait()
        el = time.time() - t0
        log.write(f"[returncode={rc} elapsed={el:.1f}s]\n")
    print(f"[returncode={rc} elapsed={el:.1f}s]")
    assert rc == 0, f"command failed with returncode {rc}"
    return el


In [ ]:
# Cell 3: train + evaluate P0 baseline
ELAPSED = run([SCRIPT, "--label_dir", LABEL_DIR, "--output_dir", OUT,
               "--test_size", "0.2", "--seed", "42", "--threshold", "0.5"])
print(f"training wall time: {ELAPSED:.1f}s ({ELAPSED/60:.1f} min)")


In [ ]:
# Cell 4: verdict
import pandas as pd
metrics = json.loads((OUT / "metrics.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
per_label = pd.read_csv(OUT / "per_label_metrics.csv", encoding="utf-8-sig")
print(per_label.to_string(index=False))

REF_PER_LABEL = {"열상": 0.804, "복통": 0.736, "호흡곤란": 0.631, "어지러움": 0.607, "고열": 0.601,
                 "전신쇠약": 0.523, "구토": 0.486, "두통": 0.409, "오심": 0.312}
f1_col = next((c for c in per_label.columns if "f1" in c.lower()), None)
lab_col = per_label.columns[0]
if f1_col is not None:
    cmp = per_label[[lab_col, f1_col]].copy()
    cmp["ref_f1"] = cmp[lab_col].map(REF_PER_LABEL)
    cmp["diff"] = (cmp[f1_col] - cmp["ref_f1"]).round(4)
    print("\nper-label F1 vs reference:")
    print(cmp.to_string(index=False))

CKPT = OUT / "mission3_tfidf_logreg.joblib"
CKPT_SHA = sha256(CKPT)
print("\nckpt:", CKPT)
print("ckpt sha256:", CKPT_SHA)

macro = float(metrics["macro_f1"])
diff = macro - REF_MACRO_F1
passed = bool(abs(diff) <= TOL and metrics["total_sample_count"] == EXPECTED_N
              and metrics["load_failure_count"] == 0)
verdict = {"macro_f1": macro, "ref": REF_MACRO_F1, "diff": diff, "tol": TOL, "pass": passed,
           "ckpt": str(CKPT), "ckpt_sha256": CKPT_SHA, "repo_commit": HEAD,
           "script_sha256": SCRIPT_SHA, "micro_f1": metrics.get("micro_f1"),
           "train_sample_count": metrics.get("train_sample_count"),
           "validation_sample_count": metrics.get("validation_sample_count"),
           "total_sample_count": metrics.get("total_sample_count"),
           "load_failure_count": metrics.get("load_failure_count"),
           "elapsed_sec": round(ELAPSED, 1)}
(OUT / "verdict.json").write_text(json.dumps(verdict, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(verdict, ensure_ascii=False, indent=2))
print(f"\nVERDICT: {'PASS' if passed else 'FAIL'}  macro_f1={macro:.5f} ref={REF_MACRO_F1} diff={diff:+.5f} tol={TOL}")


In [ ]:
# Cell 5: pip freeze
import re
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True,
                        text=True, encoding="utf-8", errors="replace").stdout
(OUT / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
print("wrote", OUT / "pip_freeze.txt", f"({len(freeze.splitlines())} packages)")
for line in freeze.splitlines():
    if re.search(r"scikit-learn|numpy|pandas|scipy|joblib|iterative", line, re.I):
        print(line)
